In [1]:
%load_ext autoreload
%autoreload 2

# Credit Risk Assessment

In [2]:
import pandas as pd
import numpy as np

import src.utils as utils

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Modeling

### Model Baseline

In [3]:
df_train = pd.read_parquet('input/df_train_02.parquet')
X_train = df_train.drop(columns=['loan_status'])
y_train = df_train['loan_status']

In [4]:
df_test = pd.read_parquet('input/df_test_02.parquet')
X_test = df_test.drop(columns=['loan_status'])
y_test = df_test['loan_status']

In [6]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier

dummy_classifier = DummyClassifier(strategy='most_frequent', random_state=29)
dummy_classifier.fit(X_train, y_train)
y_pred_dummy = dummy_classifier.predict(X_test)
y_pred_proba_dummy = dummy_classifier.predict_proba(X_test)[:, 1]
print("--- Dummy Classifier (Baseline) ---")
print(classification_report(y_test, y_pred_dummy))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_dummy):.4f}\n")

--- Dummy Classifier (Baseline) ---
              precision    recall  f1-score   support

           0       0.78      1.00      0.88     37346
           1       0.00      0.00      0.00     10437

    accuracy                           0.78     47783
   macro avg       0.39      0.50      0.44     47783
weighted avg       0.61      0.78      0.69     47783

ROC-AUC Score: 0.5000



### Fitting Models and Results

In [22]:
from sklearn.model_selection import GridSearchCV

def model_comparison_report(latest_model: GridSearchCV,
                            dummy_clf: DummyClassifier,
                            X_test, y_test):
    # get selected features
    best_pipeline = latest_model.best_estimator_
    preprocessor = best_pipeline.named_steps['preprocessing']
    feature_names = preprocessor.get_feature_names_out()
    feature_selector = best_pipeline.named_steps['feature_selection']
    selected_mask = feature_selector.get_support()
    selected_feature_names = feature_names[selected_mask].tolist()
    print(f'Selected features ({len(selected_feature_names)} total):')
    print(f'{selected_feature_names}')

    y_pred_rf = latest_model.predict(X_test)
    y_proba_rf = latest_model.predict_proba(X_test)[:, 1]
    y_pred_proba_dummy = dummy_clf.predict_proba(X_test)[:, 1]

    print("\n--- Perbandingan Skor ROC-AUC ---")
    print(f"Dummy Classifier (Baseline): {roc_auc_score(y_test, y_pred_proba_dummy):.4f}")
    print(f"Random Forest (Tuned): {roc_auc_score(y_test, y_proba_rf):.4f}")

    print("\n--- Laporan Klasifikasi RandomForest (Test Set) ---")
    print(classification_report(y_test, y_pred_rf, target_names=['Good Loan (0)', 'Bad Loan (1)']))

#### Random Forest

In [23]:
latest_rf_grid = utils.load_model('models/rf_grid_2025_11_29_07_47_00.joblib')
model_comparison_report(latest_rf_grid, dummy_classifier, X_test, y_test)

Model loaded from: models/rf_grid_2025_11_29_07_47_00.joblib
Selected features (16 total):
['num__loan_amnt', 'num__sub_grade', 'num__annual_inc', 'num__verification_status', 'num__dti', 'num__delinq_2yrs', 'num__inq_last_6mths', 'num__open_acc', 'num__revol_bal', 'num__revol_util', 'num__total_acc', 'num__tot_cur_bal', 'num__total_rev_hi_lim', 'num__term', 'num__emp_length', 'num__credit_history_age']

--- Perbandingan Skor ROC-AUC ---
Dummy Classifier (Baseline): 0.5000
Random Forest (Tuned): 0.7089

--- Laporan Klasifikasi RandomForest (Test Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.79      0.99      0.88     37346
 Bad Loan (1)       0.63      0.05      0.09     10437

     accuracy                           0.79     47783
    macro avg       0.71      0.52      0.49     47783
 weighted avg       0.75      0.79      0.71     47783



In [24]:
latest_rf_grid = utils.load_model('models/rf_grid_2025_11_29_10_00_25.joblib')
model_comparison_report(latest_rf_grid, dummy_classifier, X_test, y_test)

Model loaded from: models/rf_grid_2025_11_29_10_00_25.joblib
Selected features (16 total):
['num__loan_amnt', 'num__sub_grade', 'num__annual_inc', 'num__verification_status', 'num__dti', 'num__delinq_2yrs', 'num__inq_last_6mths', 'num__open_acc', 'num__revol_bal', 'num__revol_util', 'num__total_acc', 'num__tot_cur_bal', 'num__total_rev_hi_lim', 'num__term', 'num__emp_length', 'num__credit_history_age']

--- Perbandingan Skor ROC-AUC ---
Dummy Classifier (Baseline): 0.5000
Random Forest (Tuned): 0.6905

--- Laporan Klasifikasi RandomForest (Test Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.81      0.89      0.85     37346
 Bad Loan (1)       0.42      0.27      0.33     10437

     accuracy                           0.76     47783
    macro avg       0.62      0.58      0.59     47783
 weighted avg       0.73      0.76      0.74     47783



In [25]:
latest_rf_grid = utils.load_model('models/rf_grid_2025_11_29_10_54_02.joblib')
model_comparison_report(latest_rf_grid, dummy_classifier, X_test, y_test)

Model loaded from: models/rf_grid_2025_11_29_10_54_02.joblib
Selected features (12 total):
['num__loan_amnt', 'num__sub_grade', 'num__annual_inc', 'num__dti', 'num__open_acc', 'num__revol_bal', 'num__revol_util', 'num__total_acc', 'num__tot_cur_bal', 'num__total_rev_hi_lim', 'num__emp_length', 'num__credit_history_age']

--- Perbandingan Skor ROC-AUC ---
Dummy Classifier (Baseline): 0.5000
Random Forest (Tuned): 0.6932

--- Laporan Klasifikasi RandomForest (Test Set) ---
               precision    recall  f1-score   support

Good Loan (0)       0.86      0.67      0.75     37346
 Bad Loan (1)       0.34      0.61      0.44     10437

     accuracy                           0.66     47783
    macro avg       0.60      0.64      0.60     47783
 weighted avg       0.75      0.66      0.68     47783



#### AdaBoost